# From Scaling Laws to Self-Improving Agents
*How LLM scaling, RLHF, and test-time compute gave rise to reasoning models — and how those reasoning models are becoming the building blocks of agentic workflows*

# Scaling Laws for Large Language Models
Since GPT-3, one of the most consistent findings in language modeling has been that increasing model scale reliably improves performance. Earlier models like BERT and T5 were capable, but performance jumped substantially once parameter counts, data, and compute all scaled up together. This relationship is formalized as **scaling laws** — predictable curves relating scale to test loss, used to guide pre-training of base models.

## Three Axes of Scaling
| Axis | What increases | Effect on test loss |
|---|---|---|
| **Compute** | Total FLOPs spent training | Goes down |
| **Dataset size** | Amount of training data | Goes down |
| **Parameter count** | Layers/weights in the transformer | Goes down |

## Parameter Growth Over Time
BERT (340M) → GPT-2 (1.5B) → GPT-3 (175B) → PaLM (540B) → GPT-4 (estimated in the trillions) — a roughly exponential trajectory from 2018 to 2024. By ~2024, pure parameter scaling began showing signs of saturation.


# Few-Shot and Zero-Shot Learning
As language models scale, they gain the ability to perform new tasks without being fine-tuned on them — simply by being given a prompt, and optionally a few examples. Before this, adapting a model to a new task required fine-tuning on a domain-specific dataset.

## Zero-Shot vs. Few-Shot
- **Zero-shot:** the model is given only a task description (e.g., *"Translate English to French"*) and must produce the correct answer with no examples.
- **Few-shot:** the model is given the task description plus a handful of worked examples, then asked to generalize to a new instance (e.g., shown a few English→French translations, then asked to translate "cheese").

This behavior showed up consistently across a wide range of tasks in GPT-3, PaLM, and similar models, and set the stage for reasoning-focused innovations like chain-of-thought prompting.


# Emergent Behavior in Large Models
Beyond the predictable improvements described by scaling laws, larger models also display **emergent behaviors** — new capabilities that appear suddenly at a certain scale rather than improving gradually. These couldn't be anticipated from smaller models; they only became visible once sufficiently large models were actually trained and evaluated.

Examples of emergent, scale-gated abilities:
- Chain-of-thought reasoning (see next section)
- Modular arithmetic
- Word unscrambling

In each case, performance stays flat (near-random) as models grow — until a certain scale threshold, at which point capability jumps sharply. This is a key reason frontier labs keep pushing scale even beyond what's needed for predictable loss reduction.


# Chain of Thought (CoT) Reasoning
Chain-of-thought prompting means showing the model not just a question and its final answer, but the reasoning steps used to get there. A model shown only final answers has to guess how to get there; a model shown the reasoning process can learn to replicate that process on new, unseen problems.

## Worked Example
**Question:** Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many does he have now?

| Prompting style | What the model sees |
|---|---|
| Standard | Question → Answer ("11") |
| Chain-of-thought | Question → *"Roger started with 5 balls. Two cans of 3 tennis balls is 6. 5 + 6 = 11."* → Answer |

> The tennis-ball example is trivially easy for modern models (any ~1B parameter model solves it without CoT today) — its value here is illustrating *how* the technique works, not demonstrating current difficulty.

## Chain of Thought Is Scale-Dependent
Tested across LaMDA, GPT, and PaLM on math benchmarks: small models (~7–8B parameters) see no benefit from chain-of-thought, while larger models show significant gains from it.


# From ChatGPT's Launch to the Modern Training Pipeline
ChatGPT launched in November 2022 and became one of the fastest-adopted software products ever (reaching 1 million users in about 5 days). Its leap over GPT-3 wasn't just more scale — it came from additional training stages layered on top of a pre-trained base model: fine-tuning on higher-quality data, **instruction tuning**, and **reinforcement learning from human feedback (RLHF)**.

## The Pipeline, Stage by Stage
1. **Pre-training** — train the model to predict the next token over broad, large-scale text data (web text, books, etc.). Compute-intensive but conceptually simple — no curated preference data required.
2. **Fine-tuning on higher-quality data** — same next-token-prediction objective as pre-training, run on much higher-quality curated sources (books, curated essays) — data companies often pay significant sums to license.
3. **Instruction tuning** — covered in detail below.
4. **RLHF** — covered in detail below.

Pre-training gives raw capability; the later stages are what make a model actually usable, instruction-following, and aligned with what people want from it.


# Instruction Tuning
Instruction tuning shows the model paired examples of instructions/questions and their correct answers, so it learns to actually follow a question and answer it — rather than just continuing text in a plausible way. The training data is typically a mix of human-generated data (following role templates) and synthetic data.

## Two Flavors of Instruction Data
- **Direct Q&A pairs** — e.g., *"What is the boiling point of nitrogen?"* → the answer.
- **Chain-of-thought fine-tuning** — the pair also includes the reasoning process used to walk through to the answer, not just the final label.

The quality and generality of this instruction dataset has an outsized impact on the quality of the resulting model — a lot of effort goes into building it well. After this stage, a model starts behaving like the assistants we interact with today: answering questions, going back and forth in conversation, and reasoning through a process to reach an answer.


# RLHF: Reinforcement Learning from Human Feedback
RLHF is the step after instruction tuning that further shapes a model's behavior using **human preferences** rather than fixed labeled answers.

## How It Differs From Instruction Tuning
Instead of a supervised prompt-and-label dataset, RLHF builds a **reward model** from human judgments:
1. Companies pay humans (sometimes domain experts, sometimes general raters) to compare answers generated by the model for the same question.
2. Raters judge which answer is better along some criterion.
3. These comparisons train a **reward model** that stands in for a human judge.
4. The reward model is then used to guide the LLM's parameters toward generating outputs the reward model scores highly.

## Reward Types
A reward model isn't limited to one criterion — it can be trained to reflect correctness, helpfulness, specificity, harmlessness, or a weighted combination of these, depending on what behavior the model builders want to prioritize.

Together, pre-training → fine-tuning on high-quality data → instruction tuning → RLHF were the core components that took models from GPT-3-era capability to something like ChatGPT — a substantial jump in usability and perceived capability, even without changing the underlying scaling recipe.


# Inference-Time Scaling: A New Frontier
Pre-training and fine-tuning were the dominant levers for making models better — until about a year and a half ago, when it became clear that **inference-time compute** is also a frontier for capability, independent of any change to the model's parameters.

## Large Language Monkeys
**Paper:** [arxiv.org/abs/2407.21787](https://arxiv.org/abs/2407.21787)

Named after the *infinite monkey theorem* (a monkey typing forever will eventually produce the works of Shakespeare), this line of work treats the LLM as the monkey:

1. Ask the model to solve the same problem **many times** (parallel sampling), instead of just once.
2. Use a **verifier or selection mechanism** to pick the correct response out of the many generated (e.g., unit tests for code generation).
3. Output the selected correct response as the final answer.

This works because model generation isn't deterministic — there's inherent variance in responses, and **temperature** can be used to control/encourage that variance.

## Results
Testing on math and coding benchmarks, sample count was scaled from 1 (the normal case) up to 10,000 per problem, measuring *coverage* — the fraction of problems solved by at least one sample:
- With a single sample, smaller open models (e.g., 3B/7B-class) performed worse than GPT-4o.
- As the number of samples increased, those same smaller models **surpassed GPT-4o's single-sample performance**.

This suggests models already "know" more than what a single generation reveals — a lot of latent capability only surfaces once you sample repeatedly. For some hard problems, only 3–4 out of 10,000 samples were correct — underscoring how much scaling inference can matter for unlocking a model's real ceiling.

## Trade-offs and Open Questions
- **Latency:** parallel samples can be run concurrently, so latency is less of a concern than one might expect — but there's still a real compute-cost trade-off, which varies with problem type and difficulty.
- **Verifiers:** this approach is easiest in verifiable domains (math, code) where a checker exists. For non-verifiable domains, research directions include training an "LLM-as-judge" or learned reward functions — verification is covered in more depth separately.
- **Temperature limits:** this doesn't generalize to arbitrarily high temperature — beyond roughly 1.2, outputs degrade into gibberish. Other techniques exist to encourage response diversity without just cranking temperature.


# DeepSeek and the Fusion of Fine-Tuning with Test-Time Scaling
DeepSeek (December 2024), along with models like the o-series and Gemini Thinking, introduced a core innovation: bringing fine-tuning and test-time scaling **together** into a self-reinforcing loop.

## The Loop
1. Test-time scaling (e.g., repeated sampling) becomes an engine for generating **synthetic training data**. For math problems with known answers, the model generates many candidate solution paths leading to the correct final answer. For coding, tons of quality solution data can be generated via test-time compute.
2. This generated data becomes part of the **training set**.
3. That data is used to fine-tune the model to get better.

This fine-tuning ↔ test-time-scaling loop is the core of "thinking" and reasoning models, and is the key self-improving mechanic behind them — there's no obvious ceiling to how far this loop can push capability.

## Test-Time Scaling Also Follows a Log-Linear Law
The Large Language Monkeys work showed log-linear coverage scaling with number of samples (given a verifier). OpenAI's o1 release (September 2024) showed a similar log-linear relationship for **pass@1** accuracy on AIME (a hard math benchmark) as test-time compute increases — notably, without changing the model's parameter count at all. This was previously known for training-time scaling; seeing it hold at test time as well was a new and significant result.


# How Reasoning Models Actually "Think"
For difficult problems, thinking models — like humans facing a hard problem — spend more time and consider multiple strategies, using chain-of-thought and related techniques. Several component skills show up in these reasoning traces:

- **Problem analysis** — the model first parses what's actually being asked (e.g., understanding input/output formats before writing code).
- **Task decomposition** — breaking a task into simpler, more addressable subtasks.
- **Self-evolution / self-correction** — the model tries something, gets feedback (running tests, using a calculator, judging its own answer), and revises. This is where you see traces like *"wait, that's wrong, let me reconsider."*
- **Alternative proposals / backtracking** — if an approach isn't working, the model can abandon it and try a different one.

These skills are partly present in curated human training data, but a large part is the model acquiring them itself during fine-tuning and RL on synthetic data — generalizing well beyond what was explicitly demonstrated.

## Where Reasoning Models Outperform (and Don't)
Compared to a non-reasoning model like GPT-4o, reasoning models like o1 tend to be better at math, data analysis, and programming — but not necessarily better at tasks like personal writing or editing text.

## Open Questions
- **Is the gain from the reasoning process itself, or just from decomposition/instruction?** Likely both — the training data and RL process make the model a more generalized thinker; these component skills (analysis, backtracking, decomposition) transfer to new problems. Framed against repeated sampling: a base model can already produce multiple reasoning chains but doesn't reliably know which is correct. Reasoning training raises **pass@1** accuracy, while raw repeated sampling only raises **pass@k** (coverage).
- **Can a different, cheaper model be used just to summarize a larger model's reasoning traces?** Reasoning capability has generally scaled with model size, so the larger model's traces are typically used directly — though models tend to prefer their *own* generated traces over another (even better) model's traces, for reasons not yet fully understood.
- **Are reasoning behaviors hard-coded or learned?** Some of both — chain-of-thought and templates appear in instruction-tuning data as a starting point (bootstrapping), but models generalize far beyond their explicit training signal once fine-tuned.
- **Can sample count adapt to problem difficulty?** Not yet publicly documented, as far as could be found, but an open direction — a reward model with some notion of "how solved is this" could guide more sampling toward harder problems.


# From LLMs to Agents
Large language models as chatbots or reasoning models are still largely **single-turn** — good conversation partners, but not something that completes a real-world task end to end. What has changed recently (agentic tools like Claude Code and Deep Research) is that models can now carry out **agentic workflows** — completing multi-step, real tasks rather than just answering questions.

## What Makes Something an "Agent" (vs. a Chatbot)
An agent is given a **goal**, and then:
1. Plans steps toward that goal.
2. Interacts with an environment (possibly via **tools** external to the model).
3. Gets feedback and corrects its steps.
4. Decides when it has achieved the goal — or when to give up.
5. Often needs some form of **memory** to keep track of progress on the task.

## Where This Stands Today
Fully open-ended agent loops (plan → act → observe → replan, indefinitely) aren't fully reliable yet in most domains. What mostly exists today is closer to **structured agentic workflows** — hand-built graphs of steps, sometimes with an LLM providing evaluation/feedback at each stage. Coding and (to a good extent) deep research are the domains where fully end-to-end agentic behavior is starting to show real signs of working.


# Building Blocks of Agentic Workflows
Common components used to construct agentic systems:

- **LLM calls** — the base unit: give an instruction/input, get an output.
- **Verifiers** — check whether an output is actually correct (e.g., running unit tests against generated code). Explored in more depth separately.
- **Critics / judges** — the "LLM-as-judge" paradigm, where another LLM call evaluates or scores an output.
- **Tool calls** — e.g., web search for deep research, or querying an API for weather data.

## Orchestration Patterns
| Pattern | Description |
|---|---|
| **Prompt chaining** | Task decomposed into subtasks, chained sequentially (similar to how reasoning models decompose problems) |
| **Routing** | Complex inputs get routed to a more elaborate set of LLM calls; simpler inputs use a lighter workflow |
| **Parallelization** | Multiple LLM calls run simultaneously (e.g., deep research querying several keywords at once), then results are aggregated |
| **Orchestrator** | A central "manager" LLM plans, then dispatches subsequent LLM calls based on that plan (visible in tools like Claude Code, which shows an explicit plan) |
| **Evaluator/judge** | Instead of real-world feedback (a user, a unit test), an LLM judges the output — a useful design pattern worth practicing hands-on |
| **Verifier-based feedback** | In verifiable domains (math, code, other rule-based domains), running an actual check (e.g., unit tests) gives reliable feedback the model can use to self-correct |

These workflows require an LLM to be good at **planning**, **multi-step reasoning**, and **self-correction** — capabilities beyond what a purely reasoning-focused (but non-agentic) model demonstrates.


# Coding Agents as a Case Study
A coding agent (e.g., Claude Code) interacting with a terminal: given an instruction (e.g., "implement this test"), it navigates repositories, searches files, views/edits lines, executes terminal commands, and adapts its next action based on command output (e.g., deciding to edit a different file based on an error).

## Why This Wasn't Reliable Until Recently
This loop existed conceptually before, but only recently became reliable in practice. Discussed drivers:
- **More powerful base models**, plus **RL with verifiable rewards** — which is working well.
- **Train-time scaling continuing to pay off.**
- A **self-improvement loop**: as models get better, they can generate more reliable tests, and those tests can then be used to verify whether generated code is actually correct — creating a positive feedback cycle. (Related work: **CodeMonkeys**, [arxiv.org/abs/2501.14723](https://arxiv.org/abs/2501.14723), explores exactly this kind of test-time compute scaling for real-world software engineering tasks.)

## The Generator–Verifier Gap
Models can generate large volumes of plausible-looking content — but whether that content is actually *useful* requires a feedback loop. This is easy in domains with strong verifiers (math, code); it's much harder in domains like creative writing, where feedback is scarce and human evaluation becomes the bottleneck. Building **robust verification** remains an open, active bottleneck in this space.

## Open Question: What's Really Driving the Gains?
If pre-training already gives the model the ability to produce a correct answer somewhere among many samples, why does RL/feedback produce such a large jump in pass@1 accuracy rather than just a marginal one? There isn't a settled consensus — whether the gains mainly come from RL itself, or from the diversity already present in pre-training data, is still actively debated. Both processes appear to help, but the full picture isn't well understood yet — this is a genuinely open research area.


# Real-World Applications of Agents Today
Beyond the general agent framework, a few domains already show agents delivering real value:

## Coding Agents
Best suited to **repetitive, well-specified work**: code migrations, version upgrades, codebase restructuring, data engineering (extract/clean/transform), and data warehouse migrations. These tasks are easy to delegate because success is verifiable (tests pass or they don't) — which is exactly the property agentic loops need to self-correct reliably.

One recurring theme: even with a clear goal, the agent first has to **clarify user intent** — the task as stated is often underspecified, so figuring out what's actually being asked (and how to verify success) is itself part of the job. Models like o3 have reportedly seen enough end-to-end traces of this pattern (search → act → verify → conclude) to handle it fairly reliably.

## Customer Support
One of the most agent-augmented domains today, broken into distinct sub-problems rather than one end-to-end system:
- **Live transcription** — creates a clean meeting/call record.
- **Knowledge Assist** — lets a support agent query an LLM against a knowledge base instead of relying on manual search, surfacing the relevant article directly.
- **Smart reply** — LLM-suggested chat responses.
- **Call summaries** — used to improve the overall customer experience.

## Research Agents / Report Generation
Given a topic (e.g., "2022 Winter Olympics opening ceremony"), an agent can identify relevant references, build an outline, summarize each source, and synthesize a full-length article — a workflow that previously required manual literature review, per-paper summarization, and synthesis.

## AI Scientist
A more forward-looking use case: agents assisting scientists across idea generation, experiment iteration, and paper writeup (referencing the "AI Scientist" paper). Even though these models hallucinate, their ability to generate a large volume of ideas can surface directions outside what a domain expert might think of on their own — making them useful specifically as a brainstorming tool, even when not every idea is correct.


# Is Reasoning an Emergent or a Trained Behavior?
Chain-of-thought reasoning was originally an **emergent, discovered** behavior — not designed in. Researchers gave models hard problems, noticed that providing reasoning chains improved performance (GSM8K was the first paper to show this), and it became much more pronounced in larger models like PaLM (e.g., PaLM being able to explain jokes was seen as a notable milestone at the time).

From there, **modern reasoning models are explicitly trained to reason** — this part is no longer purely emergent; models are deliberately fine-tuned to know when heavier reasoning is needed versus when it isn't. So: chain-of-thought's *initial discovery* was emergent, but today's reasoning models are the result of intentionally reinforcing that behavior during fine-tuning.

As for whether other emergent behaviors exist: it's less about capabilities "spontaneously emerging" and more that researchers deliberately go looking for specific behaviors — planning, multi-step reasoning, self-correction/backtracking — and then reinforce them during fine-tuning once observed. Whether that still counts as "emergent" versus "trained-for" is genuinely hard to pin down.
